In [ ]:
# install and download spaCy related modules
!pip install --upgrade spacy
!python -m spacy download en_core_web_lg # using small model (sm)
!pip install wikipedia
!pip install bs4

# spaCy
import spacy
from spacy.language import Language
from spacy.tokens import Span
from spacy.matcher import PhraseMatcher
from spacy.lang.en import English
from spacy.pipeline import EntityRuler

# Google Drive
from google.colab import drive

# Firebase/Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

# Beautiful Soup
from bs4 import BeautifulSoup as BS

# Wikipedia API
import wikipedia

# general Python modules
import json
import datetime
import requests
from pprint import pprint
import re

nlp = spacy.load("en_core_web_sm")


api = "https://www.wikidata.org/w/api.php"

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 18.3 MB/s eta 0:00:00
  Attempting uninstall: spacy
    Found existing installation: spacy 3.4.4
    Uninstalling spacy-3.4.4:
      Successfully uninstalled spacy-3.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
en-core-web-sm 3.4.1 requires spacy<3.5.0,>=3.4.0, but you have spacy 3.5.0 which is incompatible.
/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-02-13 18:29:18.367251: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enab

/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/usr/local/lib/python3.8/dist-packages/spacy/util.py:877: UserWarning: [W095] Model 'en_core_web_sm' (3.4.1) was trained with spaCy v3.4 and may not be 100% compatible with the current version (3.5.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [ ]:
# remount drive, forced if needed
drive.mount("/content/gdrive/", force_remount = True)
print("Stablished access to Google Drive")

# initialize Drive path
DRIVE_PATH = "/content/gdrive/My Drive"

Mounted at /content/gdrive/
Stablished access to Google Drive


# Text Corpus from Wikipedia

In [ ]:
entity_choosen = ["William Blake","United States dollar","Bitcoin","Peercoin","Federal Reserve System"]

for entity in entity_choosen:
# parse text from a Wikipedia page, from p elements
  r = requests.get(f"https://en.wikipedia.org/wiki/{entity}")
  soup = BS(r.text, "html.parser")
  p_els = soup.find_all("p")
  text = [p.text for p in p_els]
  # print(text)
  # basic text preprocessing
  processed_text = []
  for p in text:
    p = p.replace("\n", "") # remove new line chars
    p = p.lstrip() # remove leading blank spaces
    p = p.rstrip() # remove trailing blank space
    if p == "": # ignore empty paragraphs
      continue
  # remove citation numbers [x]
    regex_wikipedia_citation = "(\[\d+(,\s?\d+|\d*-\d+)*\])"
    p= re.sub(regex_wikipedia_citation,"",p)

    processed_text.append(p)
  text = processed_text
  # print(text)

# initialize spaCY pipeline and container of sentences
  nlp = spacy.load("en_core_web_lg")
  sentences_container = []

# split text into sentences
  for index, paragraph in enumerate(text):
  # split paragraph in sentences
    sentences = [sent.text for sent in nlp(paragraph).sents]
    sentences_container.extend(sentences)
  # print(sentences_container)
# save record in JSON file
  with open(DRIVE_PATH + f"/ie_course_2022_team08/output/{entity.lower()}_context_texts_1.json", "w", encoding = "utf-8") as f:
    json.dump(sentences_container, f, ensure_ascii = False, indent = 2)
    print(f"Saved {len(sentences_container)} context sentences")



Saved 368 context sentences
Saved 224 context sentences
Saved 488 context sentences
Saved 12 context sentences
Saved 418 context sentences


# Sentences from Wiki Data

In [ ]:
entity_choosen = ["William Blake","United States dollar","Bitcoin","Peercoin","Federal Reserve System"]

for entity in entity_choosen:

# parse text content from Wikipedia article
  wikipedia_page = wikipedia.page(entity, auto_suggest=False)
  text = wikipedia_page.content

# initialize spaCY pipeline and container of sentences
  nlp = spacy.load("en_core_web_lg")
  sentences_container = []

# split text into sentences
  sentences = [sent.text for sent in nlp(text).sents]

# basic text preprocessing
  processed_text = []
  for sent in sentences:
    sent = sent.replace("\n", "") # remove new line chars
    sent = sent.lstrip() # remove leading blank spaces
    sent = sent.rstrip() # remove trailing blank space
    if sent == "": # ignore empty sentences
      continue
  # remove citation numbers [x]
  # remove citation numbers [x]
    regex_wikipedia_citation = "(\[\d+(,\s?\d+|\d*-\d+)*\])"
    sent= re.sub(regex_wikipedia_citation,"",sent)

    processed_text.append(sent)
  text = processed_text

# save record in JSON file
  with open(DRIVE_PATH + f"/ie_course_2022_team08/output/{entity.lower()}_context_texts_2.json", "w", encoding = "utf-8") as f:
    json.dump(text, f, ensure_ascii = False, indent = 2)
    print(f"Saved {len(text)} context sentences")

Saved 318 context sentences
Saved 222 context sentences
Saved 214 context sentences
Saved 12 context sentences
Saved 455 context sentences


# Creating Training Data from 5 different entities

In [ ]:
entity_choosen = ["William Blake","United States dollar","Bitcoin","Peercoin","Federal Reserve System"]
Lables= ["PERSON", "Currency","Cryptocurrency", "Cryptocurrency","ORG"]
training_data=[]

for i in range (0, len(entity_choosen)):
  
  aliases=[]
  query= entity_choosen[i]
  params = {
      "action" : "wbsearchentities",
      "language" : "en",
      "format" : "json",
      "search" : query
    }
  r = requests.get(api, params = params)
  id = r.json()['search'][0]['id']
  # url = "https://www.wikidata.org/wiki/" + id
  # result = requests.get(url)

  # fetch entity info from the Wikidata API (entity endpoint)
  api_url = f"https://www.wikidata.org/wiki/Special:EntityData/{id}.json"
  r = requests.get(api_url, params={"format": "json"})
# simplify access to root elements of JSON object
  entity_info = r.json()["entities"][f"{id}"]

# get entity aliases
  if entity_info["aliases"].get("en"):
    aliases = [a["value"] for a in entity_info["aliases"]["en"]] if entity_info["aliases"].get("en") else []

# create container of gazetteers
  alias_list = aliases

# get entity name
  if entity_info["labels"].get("en"):
    alias_list.append(entity_info["labels"]["en"]["value"])

# get last name
  last_name = entity_info["labels"]["en"]["value"].split()[-1]
  alias_list.append(last_name)

  # initialize spaCY phrase matcher (rule-based)
  matcher = PhraseMatcher(nlp.vocab, None)
# load issues as gazetteers
  patterns = [nlp.make_doc(g) for g in alias_list]
  matcher.add("gazetteers", patterns)
  # print(entity_choosen[i])

  with open(DRIVE_PATH + f"/ie_course_2022_team08/output/{entity_choosen[i].lower()}_context_texts_1.json") as f:
    text = json.load(f)

  main_text_container = []

  for index, paragraph in enumerate(text):
  # split paragraph in sentences
    sentences = [sent.text for sent in nlp(paragraph).sents]

  # instance a pipeline to process sentences individually
    disabled_pipelines = ["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer", "ner"]
    for doc in nlp.pipe(sentences, batch_size=50, disable=disabled_pipelines):
      sent = doc.text  # sentence

    # identify gazetteer contained in Doc object (text)
      gazetteers = matcher(doc)
    # convert gazetteers as spans
      gazetteers = [doc[start:end] for _, start, end in gazetteers]
    # filter overlaping matches (spans) - keep gazetteers uniqueness
      filtered_matches = spacy.util.filter_spans(gazetteers)

    # filter sentences with gazetteers occurrences
      sentence_data = []
      if len(filtered_matches):
      #  sentence_data.append({"text":sent})
       entities = []
       for m in filtered_matches:
         span = doc[m.start:m.end]  # identify span
         matched_gazetteer = span.text
         match_info = (span.start_char, span.end_char, Lables[i])
         entities.append(match_info)
       sentence_data.append({"text":sent,"entities": entities})
       main_text_container.append(sentence_data)
  
# save record in JSON file
  with open(DRIVE_PATH + f"/ie_course_2022_team08/output/training_data/{entity_choosen[i].lower()}_ner_corpus.json", "w", encoding = "utf-8") as f:
    json.dump(main_text_container, f, ensure_ascii = False, indent = 2)
    print(f"Saved {len(main_text_container)} annotated sentences")

  for test in main_text_container:
    training_data.append(test[0])

print(f"In total {len(training_data)} training sentences with five different entities") 

Saved 200 annotated sentences
Saved 119 annotated sentences
Saved 271 annotated sentences
Saved 5 annotated sentences
Saved 187 annotated sentences
In total 782 training sentences with five different entities


# Training Custom ner Model

In [ ]:
from spacy.tokens import DocBin
from tqdm import tqdm

nlp = spacy.blank("en")
doc_bin = DocBin()

In [ ]:
from spacy.util import filter_spans

for training_example in tqdm(training_data):
  text= training_example["text"]
  labels= training_example["entities"]
  doc = nlp.make_doc(text)
  ents=[]
  for start, end, label in labels:
    span = doc.char_span(start,end,label=label, alignment_mode="contract")
    if span is None:
      print("Skipping entity")
    else:
      ents.append(span)
  filtered_ents = filter_spans(ents)
  doc.ents = filtered_ents
  doc_bin.add(doc)
doc_bin.to_disk("train.spacy")

# Ceating base_config.cfg file
# https://spacy.io/usage/training#quickstart


100%|██████████| 782/782 [00:00<00:00, 1689.67it/s]


In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg

/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-01-26 21:07:25.111327: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
!python -m spacy train config.cfg --output ./ --paths.train ./train.spacy --paths.dev ./train.spacy

/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-01-26 21:07:34.421725: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
[2023-01-26 21:07:35,410] [INFO] Set up nlp object from config
INFO:spacy:Set up nlp object from config
[2023-01-26 21:07:35,422] [INFO] Pipeline: ['tok2vec', 'ner']
INFO:spacy:Pipeline: ['tok2vec', 'ner']
[2023-01-26 21:07:35,426] [INFO] Created vocabulary
INFO:spacy:Created vocabulary
[2023-01-26 21:07:38,453] [INFO] Added vectors: en_core_web_lg
INFO:spacy:Added vectors: en_core_web_lg
tcmalloc: large alloc 1233977344 bytes == 0x37fa2000 @  0x7fe0401e6680 0x7fe040206bdd 0x7fe0373b3e09 0x7fe0373b2cdf 0x7fe0373af675 0x7fe0373afe2e 0x504866 0x56bbe1 0x569d8a 0x5f60c

# Testing the Model

In [ ]:
nlp_ner = spacy.load("model-best")

doc = nlp_ner("Many of the scammers behind Bitcoin and MobileCoin ATM tokens use crypto-to-fiat exchanges to both seed their scams and launder their proceeds. PPCoin is also known as Peercoin and PPC. Blake is the one who is the founder of cryptocurrencies in USD. Federal Reserve System is the central bank for digital currencies. Also known as Federal Reserve")
for ent in doc.ents:
  print(ent,ent.label_)

Bitcoin Cryptocurrency
PPCoin Cryptocurrency
Peercoin Cryptocurrency
PPC Cryptocurrency
Blake PERSON
USD Currency
Federal Reserve System ORG
Federal Reserve ORG
